Criar uma tabela de Clientes para equipe de marketing:

- Importante somente pessoas que tenha e-mail e telefone cadastrados

In [0]:
%python
bronze_path   = 'bikestore.bronze'
silver_path   = 'bikestore.silver'
gold_path     = 'bikestore.gold'
resource_path = "/Volumes/bikestore/resource/origem"
resource_path_volume = '/Volumes/bikestore/resource/origem'

In [0]:
%python
# criar varias tabelas temporárias de forma prática

bronze_map = {
    # View_name --------------- path
    # "tmp_bronze_brands": f"{bronze_path}.brands",
    # "tmp_bronze_categories": f"{bronze_path}.categories",
    "tmp_bronze_customers": f"{bronze_path}.customers",
    # "tmp_bronze_order_items": f"{bronze_path}.order_items",
    # "tmp_bronze_orders": f"{bronze_path}.orders",
    # "tmp_bronze_products": f"{bronze_path}.products",
    # "tmp_bronze_staffs": f"{bronze_path}.staffs",
    # "tmp_bronze_stocks": f"{bronze_path}.stocks",
    # "tmp_bronze_stores": f"{bronze_path}.stores",
}

for view_name, path in bronze_map.items():
    (spark.table(path).createOrReplaceTempView(view_name))

In [0]:
select * from tmp_bronze_customers

In [0]:
describe tmp_bronze_customers

In [0]:
select
    c.customer_id,
    c.first_name,
    c.last_name,
    c.phone,
    c.email,
    c.street,
    c.city,
    c.bronze_source_file as source_customers_file
from tmp_bronze_customers c

In [0]:
select
  c.customer_id,
  c.first_name,
  c.last_name,
  c.phone,
  c.email,
  c.street,
  c.city,
  c.bronze_source_file as source_customers_file
from
  tmp_bronze_customers c
WHERE
  1 = 1
  AND C.phone IS NOT NULL -- vazio
  AND c.phone NOT IN ("NULL", "NULL ") -- texto vazio
  AND c.email IS NOT NULL
  AND c.email NOT IN ("NULL", "NULL ")

In [0]:
%python

from pyspark.sql.functions import current_timestamp, lit

current_ts = current_timestamp()

df_customers_silver = spark.sql("""
select
  c.customer_id,
  c.first_name,
  c.last_name,
  c.phone,
  c.email,
  c.street,
  c.city,
  c.bronze_source_file as source_customers_file
from
  tmp_bronze_customers c
WHERE
  1 = 1
  AND C.phone IS NOT NULL -- vazio
  AND c.phone NOT IN ("NULL", "NULL ") -- texto vazio
  AND c.email IS NOT NULL
  AND c.email NOT IN ("NULL", "NULL ")
""")\
  .withColumn("ingestion_timestamp", current_ts) \
  .withColumn("created_at", current_ts) \
  .withColumn("updated_at", lit(None).cast("timestamp"))

In [0]:
%python
#salvar em parquet como delta na camada silver
df_customers_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable(f"{silver_path}.customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bikestore.logistics.customers;

In [0]:
%sql
CREATE OR REPLACE TABLE bikestore.logistics.customers AS
SELECT * FROM bikestore.silver.customers;

In [0]:
%sql
select * from bikestore.logistics.customers;